In [ ]:
import scanpy as sc

# Load the .h5ad file
adata = sc.read_h5ad("./data/larry/postprocessed.h5ad")

# Check the contents
print(adata)

In [ ]:
import numpy as np
import scipy.sparse as sp

clone_mat = adata.obsm["X_clone"]

# If sparse, convert to dense before saving
if sp.issparse(clone_mat):
    clone_mat = clone_mat.A

np.save("./data/larry/clonotype_matrix.npy", clone_mat)


In [ ]:
from scripts.plotting import *
import numpy as np

# Subsample to ~4000 points
n_total = adata.n_obs
n_sub = 4000
rng = np.random.default_rng(0)
idx_sub = rng.choice(n_total, size=n_sub, replace=False)

# Subset coordinates, velocities, and colors
X_sub = adata.obs[["SPRING-x", "SPRING-y"]].values[idx_sub]
V_scvelo_sub = adata.obsm["velocity_emb"][idx_sub]
V_pyro_sub   = adata.obsm["velocity_pyro_emb"][idx_sub]

time_vals = adata.obs["time_info"].astype(float).values[idx_sub]
unique_times = np.sort(np.unique(time_vals))
palette = plt.cm.viridis(np.linspace(0, 1, len(unique_times)))
color_map = {t: palette[i] for i, t in enumerate(unique_times)}
cell_colors = np.array([color_map[t] for t in time_vals])

print(f"Subsampled {n_sub:,} / {n_total:,} cells for plotting")


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 7))

plot_velocity_streamplot(
    X_sub,
    V=V_scvelo_sub,
    scatter_color=cell_colors,
    title="scVelo RNA Velocity",
    grid_size=50,
    grid_density=1.0,
    stream_density=1.2,
    scatter_size=15,
    scatter_alpha=0.6,
    arrowsize=1.5,
    ax=axs[0],
    aspect="equal",
    use_cmap=False
)

plot_velocity_streamplot(
    X_sub,
    V=V_pyro_sub,
    scatter_color=cell_colors,
    title="Pyro-Velocity (Luca et al.)",
    grid_size=50,
    grid_density=1.0,
    stream_density=1.2,
    scatter_size=50,
    scatter_alpha=0.6,
    arrowsize=1.5,
    ax=axs[1],
    aspect="equal",
    use_cmap=False
)

plt.suptitle("Subsampled comparison: scVelo vs Pyro-Velocity", fontsize=16, y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# Coordinates (SPRING layout)
X = adata.obs[["SPRING-x", "SPRING-y"]].values
V_scvelo = adata.obsm["velocity_emb"]           # scVelo field
V_pyro   = adata.obsm["velocity_pyro_emb"]      # Pyro-Velocity field

time_vals = adata.obs["time_info"].astype(float).values
unique_times = np.sort(np.unique(time_vals))
palette = plt.cm.viridis(np.linspace(0, 1, len(unique_times)))
color_map = {t: palette[i] for i, t in enumerate(unique_times)}
cell_colors = np.array([color_map[t] for t in time_vals])

def plot_velocity_field(ax, X, V, title):
    speed = np.sqrt((V ** 2).sum(1))
    sc = ax.scatter(X[:, 0], X[:, 1], s=5, c=cell_colors, alpha=0.6)
    ax.quiver(
        X[:, 0], X[:, 1],
        V[:, 0], V[:, 1],
        color="k", angles="xy", scale_units="xy", scale=25, width=0.001
    )
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title)

fig, axs = plt.subplots(1, 2, figsize=(14, 7))

plot_velocity_field(axs[0], X, V_scvelo, "scVelo RNA velocity")
plot_velocity_field(axs[1], X, V_pyro,   "Pyro-Velocity (Luca et al.)")

plt.suptitle("Comparison of RNA velocity vs Pyro-Velocity on LARRY dataset", y=0.95)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# === 1. Extract clone IDs from sparse matrix ===
clone_ids = np.array(adata.obsm['X_clone'].argmax(axis=1)).flatten()
adata.obs['clone_id'] = clone_ids.astype(str)

# === 2. Find top 10 clones ===
top10 = (
    pd.Series(clone_ids)
    .value_counts()
    .head(10)
    .index
    .astype(str)
)

# === 3. Plot one UMAP per clone ===
fig, axes = plt.subplots(2, 5, figsize=(20, 8))  # grid 2x5
axes = axes.ravel()

for i, clone in enumerate(top10):
    # Make a temporary column: this clone vs others
    adata.obs['highlight'] = np.where(adata.obs['clone_id'] == clone, clone, "other")
    
    sc.pl.umap(
        adata,
        color="highlight",
        palette=["red", "lightgrey"],  # grey background, red highlight
        size=10,
        ax=axes[i],
        show=False,
        title=f"Clone {clone}"
    )

plt.tight_layout()
plt.show()


In [ ]:
# extract coordinates
x = adata.obs["SPRING-x"].values
y = adata.obs["SPRING-y"].values

# LSK marker labels
lsk_col = "Starting population"
lsk_labels = adata.obs[lsk_col].astype(str)

# Define palette
palette = {
    "Lin-Kit+Sca1+": "#1f77b4",  # blue — LSK
    "Lin-Kit+Sca1-": "#ff7f0e",  # orange — non-LSK
}

plt.figure(figsize=(6, 6))
for label, color in palette.items():
    mask = (lsk_labels == label)
    plt.scatter(
        x[mask], y[mask],
        s=8, alpha=0.3, label=label,
        c=color
    )

plt.legend(title="Starting Population", loc="best", frameon=False)
plt.xlabel("SPRING-x")
plt.ylabel("SPRING-y")
plt.title("SPRING Embedding Colored by LSK Marker")
plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# extract coordinates
x = adata.obs["SPRING-x"].values
y = adata.obs["SPRING-y"].values

# pick the time column
time_col = "time_info" if "time_info" in adata.obs.columns else "time"
time_labels = adata.obs[time_col].astype(str)

# ensure discrete order
unique_times = sorted(time_labels.unique(), key=lambda t: t.lower())
palette = ["#3b82f6", "#f59e0b", "#10b981"]  # blue, orange, green — 3 timepoints
color_map = {t: palette[i % len(palette)] for i, t in enumerate(unique_times)}
colors = [color_map[t] for t in time_labels]

plt.figure(figsize=(6, 6))
for t in unique_times:
    mask = (time_labels == t)
    plt.scatter(
        x[mask], y[mask],
        s=8, alpha=0.3, label=t,
        c=color_map[t]
    )

plt.legend(title="Time", loc="best", frameon=False)
plt.xlabel("SPRING-x")
plt.ylabel("SPRING-y")
plt.title("SPRING Embedding Colored by Time")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# === Coordinates ===
x = adata.obs["SPRING-x"].values
y = adata.obs["SPRING-y"].values

# === Cell state column ===
state_col = "state_info"  # adjust if it’s named differently
state_labels = adata.obs[state_col].astype(str)

# === Build a color map for all unique states ===
unique_states = sorted(state_labels.unique())
# automatically assign colors from matplotlib colormap
from matplotlib.cm import get_cmap
cmap = get_cmap("tab20")  # you can use "tab10", "hsv", "Spectral", etc.
colors = {s: cmap(i / len(unique_states)) for i, s in enumerate(unique_states)}

# === Plot ===
plt.figure(figsize=(8, 8))
for s in unique_states:
    mask = (state_labels == s)
    plt.scatter(
        x[mask], y[mask],
        s=8, alpha=0.6, c=[colors[s]], label=s
    )

plt.legend(title="Cell state", bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False)
plt.xlabel("SPRING-x")
plt.ylabel("SPRING-y")
plt.title("SPRING Embedding Colored by Cell State")
plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
from scipy.sparse import csr_matrix
import seaborn as sns

# clone matrix: cells x clones
X_clone = adata.obsm['X_clone']  # sparse binary matrix
states = adata.obs['state_info'].astype(str).values
unique_states = np.unique(states)
state_to_idx = {s:i for i,s in enumerate(unique_states)}

# make indicator matrix: cells × states
rows = np.arange(len(states))
cols = [state_to_idx[s] for s in states]
data = np.ones(len(states))
X_state = csr_matrix((data, (rows, cols)), shape=(len(states), len(unique_states)))

In [ ]:
# clone × state counts
clone_state_counts = X_clone.T @ X_state   # shape: clones × states

# For each clone, mark co-occurrence of states
co_occ = (clone_state_counts.T @ (clone_state_counts > 0))  # states × states
co_occ = co_occ.toarray()

# symmetric co-occurrence: both directions
co_matrix = np.minimum(co_occ, co_occ.T)

In [ ]:
# co_matrix: states × states (counts of clones spanning states)
co_matrix = co_matrix.astype(float)

# compute correlation-style normalization
diag = np.sqrt(np.diag(co_matrix))
corr_matrix = co_matrix / np.outer(diag, diag)  # elementwise division
corr_matrix[np.isnan(corr_matrix)] = 0  # handle divide-by-zero

df_corr = pd.DataFrame(corr_matrix, 
                       index=unique_states, 
                       columns=unique_states)

# plot
plt.figure(figsize=(8,6))
sns.heatmap(df_corr, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1)

plt.title("Clonotype Co-occurrence (Correlation-style)")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────────────────────────────────────
# Step 0: Define terminal vs progenitor cells
# ─────────────────────────────────────────────
all_states = adata.obs["state_info"].astype(str).unique().tolist()
term_states = [s for s in all_states if s != "Undifferentiated"]

is_terminal = adata.obs["state_info"].isin(term_states).values
is_progenitor = ~is_terminal
adata.obs["is_terminal"] = is_terminal

print("Terminal states:", term_states)
print(f"{is_terminal.sum()} terminal cells, {is_progenitor.sum()} progenitors")

# ─────────────────────────────────────────────
# Step 1: Compute clone → terminal fate probabilities
# ─────────────────────────────────────────────
Xc = adata.obsm["X_clone"].tocsr()           # (cells × clones) binary
states = adata.obs["state_info"].astype(str).values
term_index = {s: i for i, s in enumerate(term_states)}

# One-hot for terminal cells
rows = np.where(is_terminal)[0]
cols = np.array([term_index[states[i]] for i in rows])
Y_term = csr_matrix((np.ones(len(rows)), (rows, cols)),
                    shape=(adata.n_obs, len(term_states)))

# Clone × terminal counts → probabilities
C = Xc.T @ Y_term
C = C.astype(float)
clone_sum = np.asarray(C.sum(axis=1)).ravel()
clone_sum[clone_sum == 0] = 1.0
F = C.multiply(1.0 / clone_sum[:, None])      # P(fate | clone)

# ─────────────────────────────────────────────
# Step 2: Assign clone-based soft labels to cells
# ─────────────────────────────────────────────
P_clone = Xc @ F                              # (cells × terminal_states)
adata.obsm["fate_probs_clone_only"] = P_clone

# ─────────────────────────────────────────────
# Step 3: Visualize fate probabilities on UMAP
# ─────────────────────────────────────────────
P_df = pd.DataFrame(P_clone.toarray(), columns=term_states, index=adata.obs_names)

x = adata.obs["SPRING-x"].values
y = adata.obs["SPRING-y"].values

ncols = 3
nrows = int(np.ceil(len(term_states) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))

for ax, state in zip(axes.flat, term_states):
    vals = P_df[state].values
    sc = ax.scatter(
        x, y,
        c=vals,
        cmap="viridis",
        s=4,
        vmin=0, vmax=1
    )
    ax.set_title(f"Fate prob → {state}")
    ax.axis("off")
    fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

x = adata.obs["SPRING-x"].values
y = adata.obs["SPRING-y"].values
states = adata.obs["state_info"].astype(str)

# Assign consistent palette
palette = sns.color_palette("tab20", n_colors=len(states.unique()))

plt.figure(figsize=(8, 7))
sns.scatterplot(
    x=x, y=y,
    hue=states,
    s=5, linewidth=0,
    palette=palette,
    legend="brief"
)

plt.title("Cell states (state_info)")
plt.axis("off")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", markerscale=2, frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = adata.obs["SPRING-x"].values
y = adata.obs["SPRING-y"].values

is_undiff = adata.obs["state_info"] == "Undifferentiated"
P = adata.obsm["fate_probs_clone_only"].toarray()

ncols = 3
nrows = int(np.ceil(len(term_states) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))

for ax, state in zip(axes.flat, term_states):
    vals = P[:, term_states.index(state)]

    # Plot all cells in gray
    ax.scatter(x, y, c="lightgray", s=3, alpha=0.4)

    # Overlay undifferentiated cells, colored by their clonal fate bias
    sc = ax.scatter(
        x[is_undiff],
        y[is_undiff],
        c=vals[is_undiff],
        cmap="viridis",
        s=8,
        vmin=0, vmax=1
    )

    ax.set_title(f"Undifferentiated → {state}")
    ax.axis("off")
    fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import re

# ---------------------------------------------
# get numeric day labels
# ---------------------------------------------
time_col = "time_info" if "time_info" in adata.obs.columns else "time"

def get_day(x):
    m = re.search(r"(\d+)", str(x).lower())
    return int(m.group(1)) if m else np.nan

days = adata.obs[time_col].astype(str).map(get_day).values
states = adata.obs["state_info"].astype(str).values
Xc = adata.obsm["X_clone"].tocsr()

# ---------------------------------------------
# make helper function
# ---------------------------------------------
def clone_by_state(Xc, states, mask):
    """return clone×state count matrix and labels"""
    Xc_sub = Xc[mask, :]
    states_sub = states[mask]
    unique_states = np.unique(states_sub)
    rows = np.arange(len(states_sub))
    cols = np.array([np.where(unique_states == s)[0][0] for s in states_sub])
    Y_state = csr_matrix(
        (np.ones(len(rows)), (rows, cols)),
        shape=(len(states_sub), len(unique_states))
    )
    C = Xc_sub.T @ Y_state
    return C.toarray(), unique_states

# ---------------------------------------------
# build tables
# ---------------------------------------------
# day 4 + 6
is_late = np.isin(days, [4, 6])
C_late, states_late = clone_by_state(Xc, states, is_late)
print("Late table shape:", C_late.shape, "states:", list(states_late))

# day 2
is_day2 = days == 2
C_day2, states_day2 = clone_by_state(Xc, states, is_day2)
print("Day2 table shape:", C_day2.shape, "states:", list(states_day2))

# ---------------------------------------------
# preview first few rows
# ---------------------------------------------
df_late = pd.DataFrame(C_late[:, :min(5, C_late.shape[1])],
                       columns=states_late[:min(5, len(states_late))])
df_day2 = pd.DataFrame(C_day2[:, :min(5, C_day2.shape[1])],
                       columns=states_day2[:min(5, len(states_day2))])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# ==========================================================
# 1. Helper — compute dominant state & purity
# ==========================================================
def get_dominant_info(C, state_names):
    clone_totals = C.sum(axis=1)
    clone_totals[clone_totals == 0] = 1
    idx = np.argmax(C, axis=1)
    state = np.array(state_names)[idx]
    frac = C[np.arange(C.shape[0]), idx] / clone_totals
    return state, frac


# ==========================================================
# 2. Dominant states for day2 and day4+6
# ==========================================================
dom_state_day2, dom_frac_day2 = get_dominant_info(C_day2, states_day2)
dom_state_late, dom_frac_late = get_dominant_info(C_late, states_late)

cutoff = 0.5
mask_day2 = dom_frac_day2 >= cutoff
mask_late = dom_frac_late >= cutoff

plt.figure(figsize=(6, 4))
plt.plot(np.sort(dom_frac_day2)[::-1], lw=2, color="darkorange", label="Day 2")
plt.plot(np.sort(dom_frac_late)[::-1], lw=2, color="steelblue", label="Day 4 + 6")
plt.xlabel("Clone rank")
plt.ylabel("Fraction of dominant lineage")
plt.title("Clone lineage purity (Day 2 vs Day 4/6)")
plt.legend(frameon=False)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Day2 clones ≥{cutoff}: {mask_day2.sum()} / {len(mask_day2)}")
print(f"Day4+6 clones ≥{cutoff}: {mask_late.sum()} / {len(mask_late)}")


# ==========================================================
# 3. Shared clones (present both days)
# ==========================================================
has_day2 = C_day2.sum(axis=1) > 0
has_late = C_late.sum(axis=1) > 0
shared = has_day2 & has_late

dom_state_day2_s = dom_state_day2[shared]
dom_frac_day2_s  = dom_frac_day2[shared]
dom_state_late_s = dom_state_late[shared]
dom_frac_late_s  = dom_frac_late[shared]

pure_day2 = dom_frac_day2_s >= cutoff
pure_late = dom_frac_late_s >= cutoff
valid = np.where(pure_day2 & pure_late)[0]

print(f"Selected {len(valid)} clones with purity ≥{cutoff} at both day2 and day4/6.")

clone_summary_final = pd.DataFrame({
    "clone_id": np.where(shared)[0][valid],
    "day2_state": dom_state_day2_s[valid],
    "day2_purity": dom_frac_day2_s[valid],
    "day46_state": dom_state_late_s[valid],
    "day46_purity": dom_frac_late_s[valid],
}).sort_values("day46_purity", ascending=False)

# ==========================================================
# 4. Remove undifferentiated + rare final fates
# ==========================================================
exclude_fates = ["Undifferentiated", "HSPC", "MPP", "Progenitor", "Early"]
mask_diff = ~clone_summary_final["day46_state"].isin(exclude_fates)
clone_summary_final = clone_summary_final.loc[mask_diff].reset_index(drop=True)

final_fates = clone_summary_final["day46_state"].value_counts()
min_count = final_fates.get("Mast", 0)
keep_fates = final_fates[final_fates >= min_count].index.tolist()
clone_summary_final = clone_summary_final[
    clone_summary_final["day46_state"].isin(keep_fates)
].reset_index(drop=True)

print(f"\nRemaining clones after filtering: {len(clone_summary_final)}")
print("Fates kept:", keep_fates)
print("New fate distribution:")
print(clone_summary_final["day46_state"].value_counts())


# ==========================================================
# 5. Logistic regression (day2 LSK cells)
# ==========================================================
mask_day2_LSK = (days == 2) & (adata.obs["Starting population"].astype(str) == "Lin-Kit+Sca1+")
Xc_day2 = adata.obsm["X_clone"][mask_day2_LSK.values, :].tocsr()

X_day2 = adata.layers["spliced"] if "spliced" in adata.layers else adata.X
if not isinstance(X_day2, np.ndarray):
    X_day2 = X_day2.toarray()
X_day2 = X_day2[mask_day2_LSK, :]

clone_ids = clone_summary_final["clone_id"].values
fates = clone_summary_final["day46_state"].values
clone_to_fate = dict(zip(clone_ids, fates))

# Map each cell to its clone
clone_membership = np.array(Xc_day2.nonzero()[1])
cell_clone = np.full(Xc_day2.shape[0], -1)
for i, cid in zip(Xc_day2.nonzero()[0], clone_membership):
    cell_clone[i] = cid

mask_keep = np.isin(cell_clone, clone_ids)
X_cells = X_day2[mask_keep]
cell_clones = cell_clone[mask_keep]
y_cells = np.array([clone_to_fate[c] for c in cell_clones])

print(f"Using {X_cells.shape[0]} day2 LSK cells from {len(np.unique(cell_clones))} clones")

# --- standardize & evaluate ---
scaler = StandardScaler()
X_cells = scaler.fit_transform(X_cells)

kf = GroupKFold(n_splits=5)
accs = []
for tr, te in kf.split(X_cells, y_cells, groups=cell_clones):
    clf = LogisticRegression(max_iter=1000, C=0.008, solver="lbfgs")
    clf.fit(X_cells[tr], y_cells[tr])
    accs.append(accuracy_score(y_cells[te], clf.predict(X_cells[te])))

print(f"\nMean accuracy over folds: {np.mean(accs):.3f} ± {np.std(accs):.3f}")

In [ ]:
# ==========================================================
# 6b. Logistic regression baseline (Expression + Velocity)
# ==========================================================

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score


# --- features: expression + velocity ---
mask_day2_LSK = (days == 2) & (adata.obs["Starting population"].astype(str) == "Lin-Kit+Sca1+")
Xc_day2 = adata.obsm["X_clone"][mask_day2_LSK.values, :].tocsr()

# get expression + velocity matrices
X_expr = adata.layers["spliced"][mask_day2_LSK.values, :] if "spliced" in adata.layers else adata.X[mask_day2_LSK.values, :]
X_velo = adata.layers["velocity"][mask_day2_LSK.values, :] if "velocity" in adata.layers else None

# convert to dense arrays
if not isinstance(X_expr, np.ndarray):
    X_expr = X_expr.toarray()
if X_velo is not None and not isinstance(X_velo, np.ndarray):
    X_velo = X_velo.toarray()

# replace NaNs in velocity with 0 (important!)
if X_velo is not None:
    X_velo = np.nan_to_num(X_velo, nan=0.0)

# concatenate along features
X_day2_combined = np.concatenate([X_expr, X_velo], axis=1)
print(f"Combined feature matrix shape: {X_day2_combined.shape}")

# --- labels and clone mapping ---
clone_ids = clone_summary_final["clone_id"].values
fates = clone_summary_final["day46_state"].values
clone_to_fate = dict(zip(clone_ids, fates))

clone_membership = np.array(Xc_day2.nonzero()[1])
cell_clone = np.full(Xc_day2.shape[0], -1)
for i, cid in zip(Xc_day2.nonzero()[0], clone_membership):
    cell_clone[i] = cid

mask_keep = np.isin(cell_clone, clone_ids)
X_cells = X_day2_combined[mask_keep]
cell_clones = cell_clone[mask_keep]
y_cells = np.array([clone_to_fate[c] for c in cell_clones])

print(f"Using {X_cells.shape[0]} day-2 LSK cells from {len(np.unique(cell_clones))} clones")

# --- standardize features ---
scaler = StandardScaler()
X_cells = scaler.fit_transform(X_cells)

# --- GroupKFold CV ---
kf = GroupKFold(n_splits=5)
accs = []
for tr, te in kf.split(X_cells, y_cells, groups=cell_clones):
    clf = LogisticRegression(max_iter=1000, C=0.008, solver="lbfgs")
    clf.fit(X_cells[tr], y_cells[tr])
    accs.append(accuracy_score(y_cells[te], clf.predict(X_cells[te])))

print(f"\n[Expression + Velocity] Mean accuracy over folds: {np.mean(accs):.3f} ± {np.std(accs):.3f}")

In [ ]:
adata_indices = np.where(mask_day2_LSK)[0][mask_keep]

# ==========================================================
# Save cell-level information for future models
# ==========================================================
info_df = pd.DataFrame({
    "adata_index": adata_indices,
    "clone_id": cell_clones,
    "fate_label": y_cells,
})

info_df.to_csv("./data/larry/larry_day2_LSK_cellinfo.csv", index=False)
print("\nSaved cell info → 'larry_day2_LSK_cellinfo.csv'")